# Install Dependencies

In [1]:
!pip install --upgrade transformers

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.6/10.6 MB 90.5 MB/s eta 0:00:00:00:010:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 661.5/661.5 kB 35.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.5/4.5 MB 81.7 MB/s eta 0:00:00:00:01
  Attempting uninstall: hf-xet
    Found existing installation: hf-xet 1.3.0
    Uninstalling hf-xet-1.3.0:
      Successfully uninstalled hf-xet-1.3.0
  Attempting uninstall: huggingface-hub
    Found existing installation: huggingface_hub 1.4.1
    Uninstalling huggingface_hub-1.4.1:
      Successfully uninstalled huggingface_hub-1.4.1
  Attempting uninstall: transformers
    Found existing installation: transformers 5.0.0
    Uninstalling transformers-5.0.0:
      Successfully uninstalled transformers-5.0.0


# Demo Questions

In [3]:
import json

sample_QA = [
  {
    "question": "Which groups of individuals are bound by the Code of conduct at FPT Software?",
    "answer": "The Code of conduct binds (1) all staff of FPT Software, (2) staff of any accompanied organizations in which FPT Software owns more than 50 % of the authorized capital or total shares, and (3) anyone involved in operations with contractors, suppliers, and independent partners of FPT Software."
  },
  {
    "question": "What annual requirement must FPT Software staff fulfill to reconfirm their commitment to The Code?",
    "answer": "FPT Software staff must complete an annual training or exam to reconfirm their commitment to The Code."
  },
  {
    "question": "What actions must a Manager at FPT Software take to ensure staff understand and comply with the Company's Code of Conduct?",
    "answer": "A Manager at FPT Software must: (1) transmit and emphasize the importance of compliance and ethics to staff; (2) create a professional and safe environment that encourages staff to report negative behaviors and business issues; (3) evaluate each staff member’s compliance with the Code and other company regulations as part of performance assessments; (4) implement timely and appropriate interventions to prevent any risk of violating the Code; (5) respond to staff inquiries on compliance issues by researching the relevant information; and (6) seek assistance from the Senior management or the Legal, Risk and Compliance Department when the manager is unsure of how to respond."
  },
  {
    "question": "What is the maximum percentage of capital that an FPT Software employee may own in a customer, supplier, or competitor without written approval from the Legal, Risk and Compliance Department?",
    "answer": "An FPT Software employee may own up to 1 % of the capital of a customer, supplier, or competitor without written approval from the Legal, Risk and Compliance Department."
  },
  {
    "question": "What types of services does FPT Software provide globally?",
    "answer": "FPT Software provides globally a wide range of services including Smart factory, Digital platform, RPA, AI, IoT, Enterprise Mobilization, Cloud, AR/VR, Embedded System, Managed service, Testing, Platform modernization, Business Applications, Application Service, and BPO."
  }
]

with open("/kaggle/working/test.json", "w", encoding="utf-8") as f:
    json.dump(sample_QA, f, ensure_ascii=False, indent=2)

print("Saved to /kaggle/working/test.json")

Saved to /kaggle/working/test.json


# Serving SLMs for comparision

In [4]:
"""
Side-by-side model comparison UI using Gradio + Transformers.

Loads two models and lets you type questions interactively,
showing both responses side-by-side.

Usage:
    python -m src.comparison.serve_slm

    # From YAML config:
    python -m src.comparison.serve_slm --config comparison_config.yml

    # Custom models/device:
    python -m src.comparison.serve_slm \
        --model1 Qwen/Qwen3.5-4B \
        --model2 tnnanh1005/Qwen3.5-4B-SML \
        --device cpu \
        --max-tokens 512
"""

from __future__ import annotations

import argparse
import json
import time
from concurrent.futures import ThreadPoolExecutor
from pathlib import Path

import gradio as gr
import torch
from transformers import AutoProcessor, AutoModelForImageTextToText


# ─── Model loading ────────────────────────────────────────────────────────────

def load_model(model_id: str, device: str):
    """Load model and processor onto the specified device."""
    print(f"Loading {model_id} on {device}...")
    processor = AutoProcessor.from_pretrained(model_id)
    model = AutoModelForImageTextToText.from_pretrained(
        model_id,
        torch_dtype=torch.float16 if "cuda" in device else torch.float32,
    ).to(device)
    model.eval()
    print(f"  ✓ {model_id} loaded")
    return processor, model


# ─── Inference ────────────────────────────────────────────────────────────────

SYSTEM_PROMPT = (
    "You are a helpful and reliable assistant. "
    "Answer clearly, accurately, and concisely using only your own knowledge. "
    "Do not use tools, web search, APIs, or function calls. "
    "If you are unsure about something, say so honestly instead of making up information."
)


def run_inference(
    processor,
    model,
    question: str,
    max_new_tokens: int = 512,
    enable_thinking: bool = False,
) -> str:
    """Generate a response from a single model."""
    messages = [
        {"role": "system", "content": [{"type": "text", "text": SYSTEM_PROMPT}]},
        {"role": "user", "content": [{"type": "text", "text": question}]},
    ]
    inputs = processor.apply_chat_template(
        messages,
        add_generation_prompt=True,
        tokenize=True,
        return_dict=True,
        return_tensors="pt",
        enable_thinking=enable_thinking,
    ).to(model.device)

    with torch.no_grad():
        outputs = model.generate(**inputs, max_new_tokens=max_new_tokens)

    response = processor.decode(
        outputs[0][inputs["input_ids"].shape[-1]:],
        skip_special_tokens=True,
    ).strip()
    return response


# ─── QA examples loader ───────────────────────────────────────────────────────

def load_qa_examples(qa_path: str | Path | None) -> list[dict]:
    """Load QA pairs from a JSON file. Returns list of {question, answer, source_file}."""
    if qa_path is None:
        return []
    path = Path(qa_path)
    if not path.exists():
        print(f"  [warn] QA examples file not found: {path}")
        return []
    with open(path, "r", encoding="utf-8") as f:
        data = json.load(f)
    print(f"  Loaded {len(data)} example QA pairs from {path.name}")
    return data


# ─── Gradio app ───────────────────────────────────────────────────────────────

def build_app(
    model1_id: str,
    model2_id: str,
    device: str,
    max_tokens: int,
    qa_examples_path: str | Path | None = None,
):
    # Determine devices
    if device == "auto":
        if torch.cuda.is_available() and torch.cuda.device_count() >= 2:
            dev1, dev2 = "cuda:0", "cuda:1"
        elif torch.cuda.is_available():
            dev1, dev2 = "cuda:0", "cuda:0"
        elif hasattr(torch.backends, "mps") and torch.backends.mps.is_available():
            dev1, dev2 = "mps", "mps"
        else:
            dev1, dev2 = "cpu", "cpu"
    else:
        dev1, dev2 = device, device

    print(f"\nDevices: Model1={dev1}, Model2={dev2}")
    processor1, model1 = load_model(model1_id, dev1)
    processor2, model2 = load_model(model2_id, dev2)
    print("\nBoth models loaded. Starting UI...\n")

    # Load QA examples
    qa_pairs = load_qa_examples(qa_examples_path)
    # Build lookup: question -> ground truth answer
    qa_lookup = {pair["question"]: pair.get("answer", "") for pair in qa_pairs}

    def compare(question: str) -> tuple[str, str, str]:
        if not question.strip():
            return "", "", ""

        with ThreadPoolExecutor(max_workers=2) as executor:
            t0 = time.perf_counter()
            f1 = executor.submit(run_inference, processor1, model1, question, max_tokens)
            f2 = executor.submit(run_inference, processor2, model2, question, max_tokens)
            answer1 = f1.result()
            answer2 = f2.result()
            elapsed = time.perf_counter() - t0

        timing = f"Generated in {elapsed:.2f}s"
        return answer1, answer2, timing

    # Build UI
    with gr.Blocks(title="SML Model Comparison", theme=gr.themes.Soft()) as app:
        gr.Markdown("# SML Model Comparison")
        gr.Markdown(
            f"**Model A:** `{model1_id}`  \n"
            f"**Model B:** `{model2_id}`  \n"
            f"Type a question below to see both responses side-by-side."
        )

        with gr.Row():
            question_input = gr.Textbox(
                label="Your Question",
                placeholder="Type your question here...",
                lines=3,
                scale=4,
            )
            submit_btn = gr.Button("Compare", variant="primary", scale=1)

        ground_truth_display = gr.Textbox(
            label="Expected Answer (ground truth)",
            lines=3,
            interactive=False,
        )

        timing_display = gr.Textbox(label="Timing", interactive=False)

        with gr.Row():
            output1 = gr.Textbox(
                label=f"Model A: {model1_id}",
                lines=15,
                interactive=False,
            )
            output2 = gr.Textbox(
                label=f"Model B: {model2_id}",
                lines=15,
                interactive=False,
            )

        # Wire up events — Compare does NOT touch ground_truth_display
        submit_btn.click(
            fn=compare,
            inputs=[question_input],
            outputs=[output1, output2, timing_display],
        )
        question_input.submit(
            fn=compare,
            inputs=[question_input],
            outputs=[output1, output2, timing_display],
        )

        # Suggested questions — compact Examples list that also fills ground truth
        if qa_pairs:
            # Each example row: [question_text, ground_truth_text]
            example_rows = [
                [pair["question"], pair.get("answer", "")]
                for pair in qa_pairs[:10]
            ]
            # Hidden textbox carries the ground truth through gr.Examples
            gt_input = gr.Textbox(visible=False)

            def fill_from_example(question: str, gt: str):
                return question, gt

            gr.Examples(
                examples=example_rows,
                inputs=[question_input, gt_input],
                outputs=[question_input, ground_truth_display],
                fn=fill_from_example,
                run_on_click=True,
                label="Examples",
            )

    return app



In [5]:
app = build_app(
        model1_id="Qwen/Qwen3.5-4B",
        model2_id="tnnanh1005/Qwen3.5-4B-SML-ver2",
        device="auto",
        max_tokens=512,
        qa_examples_path="/kaggle/working/test.json"
    )
app.launch(server_port=7860, share=True)


Devices: Model1=cuda:0, Model2=cuda:1
Loading Qwen/Qwen3.5-4B on cuda:0...


preprocessor_config.json:   0%|          | 0.00/390 [00:00<?, ?B/s]

chat_template.jinja: 0.00B [00:00, ?B/s]

config.json: 0.00B [00:00, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

...

generation_config.json:   0%|          | 0.00/137 [00:00<?, ?B/s]

  ✓ tnnanh1005/Qwen3.5-4B-SML-ver2 loaded

Both models loaded. Starting UI...

  Loaded 5 example QA pairs from test.json


/tmp/ipykernel_57/2053910648.py:154: DeprecationWarning: The 'theme' parameter in the Blocks constructor will be removed in Gradio 6.0. You will need to pass 'theme' to Blocks.launch() instead.
  with gr.Blocks(title="SML Model Comparison", theme=gr.themes.Soft()) as app:


* Running on local URL:  http://127.0.0.1:7860
* Running on public URL: https://0c804e0efe62a0c48e.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


[transformers] Setting `pad_token_id` to `eos_token_id`:248044 for open-end generation.
[transformers] Setting `pad_token_id` to `eos_token_id`:248044 for open-end generation.
[transformers] Setting `pad_token_id` to `eos_token_id`:248044 for open-end generation.
[transformers] Setting `pad_token_id` to `eos_token_id`:248044 for open-end generation.
[transformers] Setting `pad_token_id` to `eos_token_id`:248044 for open-end generation.
[transformers] Setting `pad_token_id` to `eos_token_id`:248044 for open-end generation.
[transformers] Setting `pad_token_id` to `eos_token_id`:248044 for open-end generation.
[transformers] Setting `pad_token_id` to `eos_token_id`:248044 for open-end generation.
[transformers] Setting `pad_token_id` to `eos_token_id`:248044 for open-end generation.
[transformers] Setting `pad_token_id` to `eos_token_id`:248044 for open-end generation.
[transformers] Setting `pad_token_id` to `eos_token_id`:248044 for open-end generation.
[transformers] Setting `pad_toke